# EarningsOrderflowStudy: Microstructure Response to MSFT Earnings

This notebook performs a minute-level event study of Microsoft (MSFT) earnings announcements. We analyze the price action, volume, and orderflow proxies around the event time $t_{event}$, defined by the SEC 8-K filing timestamp.

## Methodology

<!-- TODO: YOUR INTERPRETATION HERE -->
Add 2-3 paragraphs on event-study framework, tick-rule logic, L-McD scoring, and the N=5 caveat.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add parent directory to path for src imports
sys.path.append('..')

from src.sentiment import score_lm
from src.event_study import build_event_panel, cross_event_average, cumulative_return_per_event
from src.viz import plot_all_signatures, plot_cross_event_scatter, plot_all_events_overlay

# Ensure outputs directory exists
os.makedirs('../outputs', exist_ok=True)

# 1. Load data
events_df = pd.read_parquet('../data/events.parquet')
lm_dict = pd.read_csv('../data/lm_dict.csv')

print(f"Loaded {len(events_df)} events.")

In [ ]:
# 2. Score Sentiment
sentiment_results = []
for idx, row in events_df.iterrows():
    date_str = row['announce_date'].strftime('%Y-%m-%d')
    with open(f'../data/press_releases/event_{date_str}.txt', 'r') as f:
        text = f.read()
    
    scores = score_lm(text, lm_dict)
    scores['announce_date'] = row['announce_date']
    sentiment_results.append(scores)

sent_df = pd.DataFrame(sentiment_results)
events_df = events_df.merge(sent_df, on='announce_date')
events_df.to_parquet('../data/events_augmented.parquet')

print("Sentiment scoring complete.")

In [ ]:
# 3. Build Panel and Compute Averages
panel = build_event_panel(events_df, bars_dir='../data/bars/')
avg_df = cross_event_average(panel, ['log_return', 'volume', 'cvd', 'rv5', 'spread', 'cum_return'])
per_event_cum = cumulative_return_per_event(panel)

print("Event panel and averages constructed.")

## Results: Event Signatures

The following plots show the averaged response across all 5 events. The shaded area represents $\pm 1$ standard error.

In [ ]:
plot_all_signatures(avg_df, '../outputs/event_signature.png')
display(Image(filename='../outputs/event_signature.png'))

<!-- TODO: YOUR INTERPRETATION HERE -->
Interpret the 4-panel signature. Does volume decay as expected? Is the CVD proxy revealing net pressure?

## Cross-Event Analysis

We link the news features (Sentiment and EPS Surprise) to the magnitude of the price reaction at $\tau=30$ minutes.

In [ ]:
plot_cross_event_scatter(events_df, panel, '../outputs/cross_event_scatter.png')
display(Image(filename='../outputs/cross_event_scatter.png'))

<!-- TODO: YOUR INTERPRETATION HERE -->
Which feature has higher predictive power in this small sample? Sentiment or Surprise?

## Per-Event Variation

Overlaying all events to see individual paths.

In [ ]:
plot_all_events_overlay(per_event_cum, '../outputs/all_events_overlay.png')
display(Image(filename='../outputs/all_events_overlay.png'))

## Summary Table

Final overview of the events studied.

In [ ]:
resp_30 = panel.xs(30, level='tau')['cum_return']
summary = events_df.copy()
summary['date'] = summary['announce_date'].dt.date
summary['return_30m'] = summary['date'].astype(str).map(resp_30)
summary['surprise'] = (summary['eps_actual'] - summary['eps_estimate']) / summary['eps_estimate'].abs()

cols = ['date', 'net_sentiment', 'surprise', 'return_30m']
display(summary[cols].sort_values('date', ascending=False))